# CU Boulder, CSCI 3202 - Introduction to Artificial Intelligence 

# Gil Eskayo
# Mancala Game Implementation in Python

(Run the cell below to define the class Mancala and make the game run correctly, the other cells below demonstrate how the players play against one another)

In [2]:
from games import *

import random

class Mancala(Game):
    def __init__(self, pits_per_player, stones_per_pit):
        """
        The constructor for the Mancala class defines several instance variables:

        pits_per_player: This variable stores the number of pits each player has.
        stones_per_pit: It represents the number of stones each pit contains at the start of any game.
        current_player: This variable takes the value 1 or 2, as it's a two-player game, indicating which player's turn it is.
        p1_pits_index: A list containing two elements representing the start and end indices of player 1's pits in the board data structure.
        p2_pits_index: Similar to p1_pits_index, it contains the start and end indices for player 2's pits on the board.
        p1_mancala_index and p2_mancala_index: These variables hold the indices of the Mancala pits on the board for players 1 and 2, respectively.
        """
        board = [stones_per_pit] * ((pits_per_player+1) * 2)
        self.pits_per_player = pits_per_player
        self.players = 2
        self.p1_pits_index = [0, self.pits_per_player-1]
        self.p1_mancala_index = self.pits_per_player
        self.p2_pits_index = [self.pits_per_player+1, len(board)-1-1]
        self.p2_mancala_index = len(board)-1
        
        # Zeroing the Mancala for both players
        board[self.p1_mancala_index] = 0
        board[self.p2_mancala_index] = 0

        #initial state for mancala game
        self.initial = GameState(to_move=1, utility=0, board= [stones_per_pit] * ((pits_per_player+1) * 2), moves= [1,2,3,4,5,6])


    def display(self, state):
        """
        Displays the board in a user-friendly format
        """
        player_1_pits = state.board[self.p1_pits_index[0]: self.p1_pits_index[1]+1]
        player_1_mancala = state.board[self.p1_mancala_index]
        player_2_pits = state.board[self.p2_pits_index[0]: self.p2_pits_index[1]+1]
        player_2_mancala = state.board[self.p2_mancala_index]

        print('P1               P2')
        print('     ____{}____     '.format(player_2_mancala))
        for i in range(self.pits_per_player):
            if i == self.pits_per_player - 1:
                print('{} -> |_{}_|_{}_| <- {}'.format(i+1, player_1_pits[i], 
                        player_2_pits[-(i+1)], self.pits_per_player - i))
            else:    
                print('{} -> | {} | {} | <- {}'.format(i+1, player_1_pits[i], 
                        player_2_pits[-(i+1)], self.pits_per_player - i))
            
        print('         {}         '.format(player_1_mancala))
        turn = 'P1' if state.to_move == 1 else 'P2'
        print('Turn: ' + turn)
        
    def valid_move(self, board, player, pit):
        """
        Function to check if the pit chosen by the current_player is a valid move.
        """
        
        # write your code here

        #value to return to check if valid
        valid = True

        #player 1 valid move
        if player == 1:
            #check in range
            if pit-1 not in range (0, self.pits_per_player):
            #not valid move, prompt picking new move
                valid = False
                
            #check if pit not empty
            if board[pit-1] == 0 :
                #not valid move, prompt picking new move
                valid = False

        #player 2 valid move
        else:
            if pit-1 not in range(self.pits_per_player+1, len(board)-1):
            #not valid move, prompt picking new move
                valid = False
            
            #check if pit not empty
            if board[pit-1] == 0:
                #not valid move, prompt picking new move
                valid = False

        return valid
        
    
    def result(self, state, pit):
        """
        This function simulates a single move made by a specific player using their selected pit. It primarily performs three tasks:
        1. It checks if the chosen pit is a valid move for the current player. If not, it prints "INVALID MOVE" and takes no action.
        2. It verifies if the game board has already reached a winning state. If so, it prints "GAME OVER" and takes no further action.
        3. After passing the above two checks, it proceeds to distribute the stones according to the specified Mancala rules.

        Finally, the function then switches the current player, allowing the other player to take their turn.
        """
        
        # write your code here
        previous = state.board.copy()
        board = state.board.copy()
        
        #check player turn
        
        if state.to_move == 2:
            pit = pit + self.pits_per_player + 1
        
        if self.valid_move(board, state.to_move, pit) == 1:
            move = (state.to_move, pit)
            #print (move)
            if self.terminal_test(state) ==  False:
                stones = board[pit - 1] #amount of stones to move
                      
                board[pit -1] = 0 #empty current pit
                pit_insert = pit - 1 #pit to insert into
                for i in range (stones): #iteration for each stone placed
                    pit_insert = pit_insert + 1 #moving to next pit each stone
                    if pit_insert > len(board)-1: #if we hit end of index, reset pit to insert to 0
                        pit_insert = 0
                        
                    if pit_insert == self.p1_mancala_index: #if inserting to p1 mancala, check that player is p1
                        if state.to_move != 1: #skip if not p1
                            pit_insert = pit_insert + 1
                                
                    if pit_insert == self.p2_mancala_index: #if inserting to p2 mancala, check that player is p2
                        if state.to_move != 2: #skip if not p2
                            pit_insert = 0

                    board[pit_insert] += 1 #drop stone into pit

                #if the final pit we inserted is not the mancala and was empty prior to insertion and was on the current players side
                if state.to_move == 1:
                    if pit_insert in range(0, self.pits_per_player) and board[pit_insert] == 1:
                        captured = 1 + board[len(board)-2 - pit_insert] #stones captured from your side and other side
                        board[pit_insert] = 0 #capture the stone just placed
                        board[len(board)-2 - pit_insert] = 0 #capture opponents stone from adjacent pit
                        board[self.p1_mancala_index] += captured

                if state.to_move == 2:
                    if pit_insert in range(self.pits_per_player+1, len(board)-1) and board[pit_insert] == 1:
                        captured = 1 + board[self.pits_per_player - 1 - (pit_insert - (self.pits_per_player + 1))] #stones captured from your side and other side
                        board[pit_insert] = 0 #capture the stone just placed
                        board[self.pits_per_player - 1 - (pit_insert - (self.pits_per_player + 1))] = 0  #capture opponents stone from adjacent pit
                        board[self.p2_mancala_index] += captured


                #if game ended from turn
                p1Side= 1

                p2Side = 1
         
                #check p1 side
                for i in range(0, self.pits_per_player):
                    if board[i] != 0: #non empty pit
                        p1Side = 0 #game not over
        
                #check p2 side
                for i in range(self.pits_per_player+1, len(board)-1):
                    if board[i] != 0:
                        p2Side = 0
        
                if p1Side or p2Side == 1: #game over
                    for i in range (0, self.pits_per_player): #add p1 pits to p1 mancala
                        board[self.p1_mancala_index] += state.board[i]
                        board[i] = 0
        
                    for i in range (self.pits_per_player+1, len(board)-1): #add p1 pits to p2 mancala
                        board[self.p2_mancala_index] += board[i]
                        board[i] = 0

        move = 1 if state.to_move == 2 else 2
        return GameState(to_move=move, 
                         utility= self.compute_utility2(board),
                         board= board, 
                         moves= self.next_actions(board, move))
      

    def utility(self, state, player):
        #estimated utility of move
        return state.utility if player == 1 else -state.utility 
        
    def compute_utility(self, board):
        #how we compute utility
        
        return board[self.p1_mancala_index] - board[self.p2_mancala_index] #how many more stones player1 has than player2

    def compute_utility2(self, board):
        #alternative utility that measures amount of stones captured unless it is a terminal state, then it gives a large reward
        # (VERY BAD UTILITY FUNCTION DONT USE THIS)
        p1 = 0
        p2 = 0
        for i in range (0, self.pits_per_player+1):
            p1 += board[i]
        for i in range(self.pits_per_player+1, len(board)):
            p2 += board[i]
        return p1 - p2

    def next_actions(self, board, player):
        """Locally defined next moves available"""
        valid = []
        
        if player == 1: #player 1
            for i in range(1, self.pits_per_player+1):
                if self.valid_move(board, player, i) == 1:
                    valid.append(i)

        else: #player 2
            for i in range(self.pits_per_player+2, len(board)):
                if self.valid_move(board, player, i) == 1:
                    valid.append(i+1 - (len(board) - self.pits_per_player))

        
        #returns a list of valid moves
        return valid    
    
    def actions(self, state):
        """Return a list of the allowable moves at this point."""
        valid = []
        
        if state.to_move == 1: #player 1
            for i in range(1, self.pits_per_player+1):
                if self.valid_move(state.board, state.to_move, i) == 1:
                    valid.append(i)

        else: #player 2
            for i in range(self.pits_per_player+2, len(state.board)):
                if self.valid_move(state.board, state.to_move, i) == 1:
                    valid.append(i+1 - (len(state.board) - self.pits_per_player))

        
        #returns a list of valid moves
        return valid

    def terminal_test(self, state):
        """Return True if this is a final state for the game."""

        return not self.actions(state)

    def to_move(self, state):
        """Return the player whose move it is in this state."""
        
        return state.to_move


ModuleNotFoundError: No module named 'games'

# The cell below defines the Minimax search function. By default it is set to run at a depth of 4.

In [2]:
def minmax_decision(state, game, d=4, cutoff_test=None, eval_fn=None):
    """Given a state in a game, calculate the best move by searching
    forward all the way to the terminal states. [Figure 5.3]"""

    player = game.to_move(state)

    def max_value(state, depth):
        if cutoff_test(state, depth):
            return eval_fn(state)
        v = -np.inf
        for a in game.actions(state):
            v = max(v, min_value(game.result(state, a), depth + 1))
        return v

    def min_value(state, depth):
        if cutoff_test(state, depth):
            return eval_fn(state)
        v = np.inf
        for a in game.actions(state):
            v = min(v, max_value(game.result(state, a), depth + 1))
        return v

    # Body of minmax_decision:
    cutoff_test = (cutoff_test or (lambda state, depth: depth > d or game.terminal_test(state)))
    eval_fn = eval_fn or (lambda state: game.utility(state, player))
    best_score = -np.inf
    for a in game.actions(state):
        v = min_value(game.result(state, a), 1)
        if v > best_score:
            best_score = v
            best_action = a
    return best_action
    return max(game.actions(state), key=lambda a: min_value(game.result(state, a)))

# Below runs 100 games of ai vs random player using the minmax decision function

In [3]:
wins = []
moves = 0
for i in range(100): #number of games played
    i = 0
    game = Mancala(6,4)
    state = game.initial
    while game.terminal_test(state) == False:
        move = minmax_decision(state, game)
        state = game.result(state, move)
        moves += 1
        if game.terminal_test(state) == True:
            break
        state = game.result(state, random.choice(game.actions(state)))
        moves += 1
    if state.board[game.p1_mancala_index] > state.board[game.p2_mancala_index]:
        wins.append(1)
    elif state.board[game.p2_mancala_index] > state.board[game.p1_mancala_index]:
        wins.append(2)
    else:
        wins.append("tie")

games = len(wins)
avg_moves = moves/games
p1 = wins.count(1) / games * 100
p2 = wins.count(2) / games * 100
Tie = wins.count("tie") / games * 100
print("AI: ", p1, "%")
print("P2: ", p2, "%")
print("Ties: ", Tie, "%")
print("Average moves: ", avg_moves)

AI:  99.0 %
P2:  1.0 %
Ties:  0.0 %
Average moves:  28.82


# The cell below defines the Alpha Beta search function. By default it is set to run at a depth of 4.

In [11]:
def alpha_beta_cutoff_search(state, game, d=4, cutoff_test=None, eval_fn=None):
    """Search game to determine best action; use alpha-beta pruning.
    This version cuts off search and uses an evaluation function."""

    player = game.to_move(state)

    # Functions used by alpha_beta
    def max_value(state, alpha, beta, depth):
        if cutoff_test(state, depth):
            return eval_fn(state)
        v = -np.inf
        for a in game.actions(state):
            v = max(v, min_value(game.result(state, a), alpha, beta, depth + 1))
            if v >= beta:
                return v
            alpha = max(alpha, v)
        return v

    def min_value(state, alpha, beta, depth):
        if cutoff_test(state, depth):
            return eval_fn(state)
        v = np.inf
        for a in game.actions(state):
            v = min(v, max_value(game.result(state, a), alpha, beta, depth + 1))
            if v <= alpha:
                return v
            beta = min(beta, v)
        return v

    # Body of alpha_beta_cutoff_search starts here:
    # The default test cuts off at depth d or at a terminal state
    cutoff_test = (cutoff_test or (lambda state, depth: depth > d or game.terminal_test(state)))
    eval_fn = eval_fn or (lambda state: game.utility(state, player))
    best_score = -np.inf
    beta = np.inf
    best_action = None
    for a in game.actions(state):
        v = min_value(game.result(state, a), best_score, beta, 1)
        if v > best_score:
            best_score = v
            best_action = a
    return best_action

# Below runs 100 games of the ai vs random using the alpha beta function

In [9]:
wins = []
moves = 0
for i in range(100): #number of games played
    i = 0
    game = Mancala(6,4)
    state = game.initial
    while game.terminal_test(state) == False:
        move = alpha_beta_cutoff_search(state, game)
        state = game.result(state, move)
        moves += 1
        if game.terminal_test(state) == True:
            break
        state = game.result(state, random.choice(game.actions(state)))
        moves += 1
    if state.board[game.p1_mancala_index] > state.board[game.p2_mancala_index]:
        wins.append(1)
    elif state.board[game.p2_mancala_index] > state.board[game.p1_mancala_index]:
        wins.append(2)
    else:
        wins.append("tie")

games = len(wins)
avg_moves = moves/games
p1 = wins.count(1) / games * 100
p2 = wins.count(2) / games * 100
Tie = wins.count("tie") / games * 100
print("AI: ", p1, "%")
print("P2: ", p2, "%")
print("Ties: ", Tie, "%")
print("Average moves: ", avg_moves)

AI:  100.0 %
P2:  0.0 %
Ties:  0.0 %
Average moves:  24.05


# For testing, a single game using the Alpha-Beta-search can be run below
*note that this will display every single move made

In [10]:
moves = 0
game = Mancala(6,4)
state = game.initial
while game.terminal_test(state) == False:
    move = alpha_beta_cutoff_search(state, game)
    state = game.result(state, move)
    moves += 1
    print("AI made move: ", move)
    game.display(state)
    if game.terminal_test(state) == True:
        break
    move = random.choice(game.actions(state))
    state = game.result(state, move)
    moves += 1
    print("Random made move: ", move)
    game.display(state)

print("--------------------------------------------------------------------FINAL--------------------------------------------------------------------")
game.display(state)
print("P1: ", state.board[game.p1_mancala_index])
print("P2: ", state.board[game.p2_mancala_index])
print("Moves: ", moves)

AI made move:  5
P1               P2
     ____4____     
1 -> | 4 | 4 | <- 6
2 -> | 4 | 4 | <- 5
3 -> | 4 | 4 | <- 4
4 -> | 4 | 4 | <- 3
5 -> | 0 | 5 | <- 2
6 -> |_5_|_5_| <- 1
         5         
Turn: P2
Random made move:  2
P1               P2
     ____5____     
1 -> | 4 | 5 | <- 6
2 -> | 4 | 5 | <- 5
3 -> | 4 | 5 | <- 4
4 -> | 4 | 5 | <- 3
5 -> | 0 | 0 | <- 2
6 -> |_5_|_5_| <- 1
         5         
Turn: P1
AI made move:  6
P1               P2
     ____5____     
1 -> | 4 | 5 | <- 6
2 -> | 4 | 5 | <- 5
3 -> | 4 | 6 | <- 4
4 -> | 4 | 6 | <- 3
5 -> | 0 | 1 | <- 2
6 -> |_0_|_6_| <- 1
         6         
Turn: P2
Random made move:  1
P1               P2
     ____6____     
1 -> | 4 | 6 | <- 6
2 -> | 4 | 6 | <- 5
3 -> | 4 | 7 | <- 4
4 -> | 4 | 7 | <- 3
5 -> | 0 | 2 | <- 2
6 -> |_0_|_0_| <- 1
         6         
Turn: P1
AI made move:  1
P1               P2
     ____6____     
1 -> | 0 | 6 | <- 6
2 -> | 5 | 6 | <- 5
3 -> | 5 | 7 | <- 4
4 -> | 5 | 7 | <- 3
5 -> | 0 | 0 | <- 2
6 -> |_0_|_

# Writeup
On average the AI is winning about 96% of games with the average amount of moves per game around 27-28. 
100 Games can be simulated much quicker using Alpha-Beta pruning than using Minmax decisions. While using Alpha-Beta Pruning, on my laptop it takes an average of 10 seconds to run 100 games as opposed to roughly 3 times as long at over 30 seconds to run 100 games using Minimax. This is because they are essentially the exact same algorithm, however Alpha-Beta saves time by ignoring unecessary computations and pruning nodes that are irrelevant. By doing this, we have to explore much less nodes in the state tree for each possible move and thus save on computation power and time spent doing these computations speeding up our end result.

If we increase the search depth (or the number of plies) we see that the AI gives us a better result. This is because by increaseing the search depth, the AI is evaluating more states and thus has a better pool of moves to choose from by considering another layer of moves to the game. The tradeoff for this is that simulating games takes longer because this exponentially increases the amount of moves you are evaluating before making your choice.

The AI is much better than the random move generator because it makes decisions based off of the current game state (and resulting states of moves from each state) with knowledge of which moves can lead to the best outcome for itself while simultanouesly leaving its opponent with the worst moves for them. As seen, this yields a better result for about 96% of games. 

Using the helper function calculate_utility2 which is a unique utility function that I created, we get around a 98% win rate for the AI. This utility is similar to the original that bases itself on the amount of stones the player has captured over the amount of stones the opponent has captured. However with this utility funciton, we also take into consideration how many stones are on each players side since that affects the amount of stones that player is able to capture as we approach the end state of the game. Adding how many stones the player has captured and how many stones are on their side of the board compared to the same evaluation for the opponent thus yields a higher win rate than just considering the amount of stones that the player has captured over the opponent by about 2%.

With this new helper function and running the alpha-beta search with a depth of 10 the 100 games took around 18 minutes to simulate and resulted in a 100% win rate for the AI.